# Phase 2 - end-to-end fusion on Colab

The Phase 2 ladder was measured with **cached** (frozen) sequence embeddings, which
keeps the fusion effect separable from an adaptation effect. This notebook runs the
**end-to-end** setting instead: the ChemBERTa LoRA adapters train alongside the
graph and descriptor encoders.

**The question it answers.** With cached views, a 16,513-parameter gate matched the
1,169,793-parameter `proposed` architecture on 7 of 8 datasets. Does that still hold
when the sequence view can adapt?

**All five variants.** `concat` and `gated` as baselines, `xattn` (~MvMRL) and
`bilinear` (~KROVEX) as the individual mechanisms, `proposed` as their composition.

If you already ran `concat`, `gated` and `proposed`, **`--resume` skips them** and
only `xattn` and `bilinear` are trained -- roughly **7-8 hours** rather than 18.

Why these two are worth the time: end to end, `proposed` beats `gated` on BACE and
FreeSolv, and without the individual mechanisms there is no way to say which of
cross-attention or the bilinear term is responsible. The cached ladder answers that
question; this completes the same answer under adaptation.

No free-tier session lasts this long, and it does not need to - each sitting
continues where the last stopped. Run the cells, let it go until Colab cuts you off,
then come back and run cells 1-6 again.


## 1. Check you actually got a GPU


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('NO GPU -- Runtime > Change runtime type > T4 GPU, then re-run.')


## 2. Connect your Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Point at the fusion bundle

This is a **different, larger bundle** than the Phase 1c one: fusion reads molecular
graphs and descriptors as well as tokens, so it is ~18 MB rather than 4 MB. Build it
with `python -m scripts.make_colab_bundle --preset fusion --out mpp_fusion_bundle.zip`
and upload that.

Results go to a **separate folder** from Phase 1c, so the two runs cannot overwrite
each other's archives.


In [ ]:
BUNDLE = '/content/drive/MyDrive/mpp_fusion_bundle.zip'  # edit if elsewhere
OUTDIR = '/content/drive/MyDrive/mpp_phase2'

import os
assert os.path.exists(BUNDLE), f'Not found: {BUNDLE} -- check path, re-run.'
os.makedirs(OUTDIR, exist_ok=True)
print('bundle:', round(os.path.getsize(BUNDLE)/1e6, 1), 'MB')


## 4. Unpack and install

**PyTorch Geometric is needed here**, unlike the Phase 1c notebook - the fusion
model reads the graph view. Recent PyG runs on native PyTorch scatter operations, so
the compiled `torch-scatter` / `torch-sparse` extensions are not required.

`torchao` is removed for the same reason as before: `peft` probes it while placing
LoRA adapters and its probe raises, rather than returning False, when the installed
version is older than 0.16.


In [ ]:
import zipfile, os

WORK = '/content/mpp'
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(WORK)
os.chdir(WORK)

!pip -q install peft accelerate torch_geometric
!pip -q uninstall -y torchao

import peft, torch_geometric
print('peft', peft.__version__, '| torch_geometric', torch_geometric.__version__)


## 5. Keep results on Drive


In [ ]:
import os

os.makedirs(f'{OUTDIR}/runs', exist_ok=True)
os.makedirs('results', exist_ok=True)
if not os.path.islink('results/runs'):
    if os.path.exists('results/runs'):
        import shutil; shutil.rmtree('results/runs')
    os.symlink(f'{OUTDIR}/runs', 'results/runs')
print('results/runs ->', os.path.realpath('results/runs'))


## 6. Smoke test - do not skip this

One dataset, one epoch, the heaviest variant. It checks that PyG, peft and the
three-view batching all work together before an 11-hour run starts.

It should print a `valid rmse=` line. The number is meaningless at one epoch.


In [ ]:
!python -m src.data.materialize --variant deepchem --artifacts tok ecfp graphs desc
!python -m src.train.train_fusion --mode proposed --tag _smoke --seq lora --datasets freesolv --epochs 1 --patience 1 --device cuda
!rm -f results/metrics/*_smoke*.csv results/preds/*_smoke*.npy
!rm -f models/*_smoke*.pt


## 7. Train

`--seq lora` is what makes this end-to-end: the sequence encoder's adapters train
instead of a cached vector being read.

`--resume` is doing real work here. Free sessions will not last 11 hours, so expect
to run this cell across several sittings; each one skips what is already finished.

Progress prints per split. If Colab disconnects mid-variant, that variant restarts
from the beginning - work is archived per (split, tag), not per epoch.


In [ ]:
!python -m scripts.run_view_multiseed --tags fuse_concat fuse_gated fuse_xattn fuse_bilinear fuse_proposed --variants deepchem seed0 seed1 seed2 seed3 seed4 --artifacts tok ecfp graphs desc --seq lora --device cuda --tag-suffix e2e --resume --restore none


## 8. Check what finished

Each split should reach 16 files per tag: 8 datasets x (valid + test).


In [ ]:
import glob
tags = ['fuse_concat_e2e','fuse_gated_e2e','fuse_xattn_e2e',
        'fuse_bilinear_e2e','fuse_proposed_e2e']
for v in ['deepchem','seed0','seed1','seed2','seed3','seed4']:
    counts = [len(glob.glob(f'{OUTDIR}/runs/{v}/metrics/*_{t}_*.csv')) for t in tags]
    done = all(c == 16 for c in counts)
    print(v, dict(zip([t.replace('fuse_','') for t in tags], counts)),
          'ok' if done else 'incomplete - re-run cell 7')


## 9. Bring the results home

Download, then on your own machine, from the project root:

```
python -m scripts.merge_colab_results ~/Downloads/phase2_results.zip --dry-run
```

Use that script rather than unzipping by hand - extracting into `results/runs/` with
the wrong root nests your local working directories inside it, which reads as every
tracked result file having been deleted.

These runs are archived as `fuse_*_e2e`, so they cannot overwrite the cached ladder
-- both sets sit side by side and can be compared directly.


In [ ]:
import shutil, os
out = shutil.make_archive('/content/phase2_results', 'zip', f'{OUTDIR}/runs')
print('wrote', out, round(os.path.getsize(out)/1e6, 2), 'MB')
from google.colab import files
files.download(out)
